# 05: Time Series Analysis (Learning Outcome 5)

**Learning Outcome 5 (LO5):** Time series forecasting and trend analysis

**Methods:**
- Exponential smoothing (Holt-Winters)
- Trend analysis
- MAPE evaluation

# Load data
df = pd.read_csv('data/processed/final_dataset.csv')
yield_series = df.sort_values('year')[['year', 'yield_kg_ha']].dropna()

print(f"Yield Time Series ({len(yield_series)} observations)")
print(yield_series)

# Extract values
y = yield_series['yield_kg_ha'].values
years = yield_series['year'].values

# Holt-Winters exponential smoothing
def exponential_smoothing(data, alpha=0.3, forecast_steps=5):
    """Simple exponential smoothing with trend"""
    level = data[0]
    trend = data[1] - data[0] if len(data) > 1 else 0
    
    forecasts = []
    
    for t in range(len(data)):
        prev_level = level
        level = alpha * data[t] + (1 - alpha) * (level + trend)
        trend = alpha * (level - prev_level) + (1 - alpha) * trend
    
    # Generate forecasts
    for h in range(1, forecast_steps + 1):
        forecasts.append(level + h * trend)
    
    return forecasts

# Apply smoothing
forecasts = exponential_smoothing(y_train, alpha=0.35, forecast_steps=len(y_test))

print(f"\nForecasts for test period:")
for year, forecast in zip(years_test, forecasts):
    print(f"  {year:.0f}: {forecast:.2f} kg/ha")

# Visualization
fig, ax = plt.subplots(figsize=(14, 6))

# Historical data
ax.plot(years_train, y_train, 'o-', linewidth=2.5, markersize=7, label='Training Data', color='steelblue')

# Test actual
ax.plot(years_test, y_test, 'o-', linewidth=2.5, markersize=7, label='Test Data (Actual)', color='green')

# Forecasts
ax.plot(years_test, forecasts, 's--', linewidth=2.5, markersize=7, label='Forecasts', color='red')

# Shaded forecast region
ax.axvspan(years_test[0]-0.5, years_test[-1]+0.5, alpha=0.1, color='gray')

ax.set_xlabel('Year', fontsize=12, fontweight='bold')
ax.set_ylabel('Yield (kg/ha)', fontsize=12, fontweight='bold')
ax.set_title(f'Rice Yield Time Series Forecasting (MAPE={mape*100:.2f}%)', 
            fontsize=13, fontweight='bold')
ax.legend(fontsize=11, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/graphs/05_time_series_forecast.png', dpi=300, bbox_inches='tight')
plt.show()

# Summary table
summary_df = pd.DataFrame({
    'Year': years_test,
    'Actual Yield (kg/ha)': y_test,
    'Forecast (kg/ha)': [f'{f:.2f}' for f in forecasts],
    'Error (kg/ha)': [f'{y-f:.2f}' for y, f in zip(y_test, forecasts)],
    'Error (%)': [f'{abs((y-f)/y*100):.2f}%' for y, f in zip(y_test, forecasts)]
})

print("\n=== Forecast Results ===")
print(summary_df.to_string(index=False))